# Day 21 — Historical
### #30DayChartChallenge | April 2026

**230 Years of Lead Pollution in the Arctic.** A single Greenland ice core (ACT2, 66°N 45.2°W) holds a continuous annual record of atmospheric lead deposition from 1772 to 2003. Against a late-18th-century baseline of ~0.010 ng Pb per gram of ice, coal burning had already pushed Arctic concentrations to ~11× pre-industrial by the early 1900s — before tetraethyl lead (1923) ever entered a gas tank. The 1950 spike hit 0.42 ng/g (44× baseline). After the US leaded-gasoline phase-out (1976–1996), concentrations fell to ~3× pre-industrial — reduced by about 85% from peak, but never restored.

**Data:** [NOAA/WDS Paleoclimatology — McConnell & Edwards 2008](https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=noaa-icecore-6177), *Coal burning leaves toxic heavy metal legacy in the Arctic*, PNAS 105(34): 12140–12144, doi:10.1073/pnas.0803564105. Data Contribution Series 2008-079.  
**Author:** Sharfudeen Yasar Arafath

In [ ]:
# — packages ------------------------------------------------------------------

library(ggplot2)
library(dplyr)
library(showtext)
library(sysfonts)
library(scales)
library(zoo)

In [ ]:
# — fonts & display size ------------------------------------------------------

font_add_google("Outfit", "outfit")
font_add_google("Roboto Condensed", "roboto_condensed")
font_add_google("JetBrains Mono", "jetbrains")
showtext_auto()
showtext_opts(dpi = 300)

options(repr.plot.width = 13, repr.plot.height = 10, repr.plot.res = 300)

In [ ]:
# — read data & compute epoch means -------------------------------------------
# Clean CSV built from the NOAA ACT2 metals file; two columns, annual means.

df <- read.csv("../../data/day_21/pb_greenland_annual.csv",
               stringsAsFactors = FALSE) %>%
  arrange(year) %>%
  mutate(roll10 = zoo::rollmean(lead_ng_g, k = 10,
                                fill = NA, align = "center"))

pre_industrial <- df %>% filter(year >= 1772, year <= 1800) %>%
  summarise(mean = mean(lead_ng_g)) %>% pull(mean)
early_1900s    <- df %>% filter(year >= 1900, year <= 1920) %>%
  summarise(mean = mean(lead_ng_g)) %>% pull(mean)
mid_century    <- df %>% filter(year >= 1940, year <= 1970) %>%
  summarise(mean = mean(lead_ng_g)) %>% pull(mean)
recent         <- df %>% filter(year >= 1995, year <= 2002) %>%
  summarise(mean = mean(lead_ng_g)) %>% pull(mean)
peak_row       <- df %>% slice_max(lead_ng_g, n = 1)

cat("Pre-industrial (1772\u20131800):", round(pre_industrial, 4), "ng/g\n")
cat("Early 1900s   (1900\u20131920):", round(early_1900s, 4), "ng/g (",
    round(early_1900s / pre_industrial, 1), "x pre-ind)\n", sep = "")
cat("Mid-century   (1940\u20131970):", round(mid_century, 4), "ng/g (",
    round(mid_century / pre_industrial, 1), "x pre-ind)\n", sep = "")
cat("Recent        (1995\u20132002):", round(recent, 4), "ng/g (",
    round(recent / pre_industrial, 1), "x pre-ind)\n", sep = "")
cat("Peak year:", peak_row$year, "at", round(peak_row$lead_ng_g, 4), "ng/g (",
    round(peak_row$lead_ng_g / pre_industrial, 1), "x pre-ind)\n", sep = "")

In [ ]:
# — build the plot ------------------------------------------------------------

bg <- "#0a0e17"; txt <- "#E6EDF3"; txt_dim <- "#8B949E"
txt_cap <- "#484F58"; grid_col <- "#1a2030"
col_line <- "#fbbf24"; col_roll <- "#f97316"

epochs <- data.frame(
  xmin = c(1772, 1860, 1923, 1976),
  xmax = c(1860, 1923, 1976, 2003),
  label = c("Pre-industrial", "Coal Era", "Leaded-Gasoline Era", "Phase-out"),
  fill  = c("#1e3a8a22", "#7f1d1d22", "#b4530922", "#14532d22")
)

events <- data.frame(
  year  = c(1815, 1860, 1923, 1970, 1976, 1996),
  label = c("Tambora\neruption",
            "Coal-burning\nsource signal begins",
            "Tetraethyl lead\nintroduced (US)",
            "US Clean Air\nAct",
            "US leaded-gas\nphase-out starts",
            "US phase-out\ncomplete")
)

p <- ggplot(df, aes(x = year, y = lead_ng_g)) +
  geom_rect(data = epochs,
            aes(xmin = xmin, xmax = xmax, ymin = 0.003, ymax = 0.6,
                fill = I(fill)),
            inherit.aes = FALSE, alpha = 1) +
  geom_text(data = epochs,
            aes(x = (xmin + xmax) / 2, y = 0.52, label = label),
            inherit.aes = FALSE, family = "outfit", fontface = "bold",
            size = 3.8, color = "#c7d2fe") +

  geom_hline(yintercept = pre_industrial, color = "#94a3b8",
             linetype = "dashed", linewidth = 0.35) +
  annotate("text", x = 2000, y = pre_industrial * 0.78,
           label = sprintf("Pre-industrial baseline: %.3f ng/g", pre_industrial),
           family = "jetbrains", size = 3, color = "#94a3b8", hjust = 1) +

  geom_step(color = col_line, linewidth = 0.55, alpha = 0.85) +
  geom_line(aes(y = roll10), color = col_roll, linewidth = 1.3) +

  annotate("point", x = peak_row$year, y = peak_row$lead_ng_g,
           color = "#FFFFFF", size = 2.8) +
  annotate("segment",
           x = peak_row$year, xend = peak_row$year - 10,
           y = peak_row$lead_ng_g, yend = peak_row$lead_ng_g * 0.62,
           color = "#FFFFFF", linewidth = 0.3) +
  annotate("text",
           x = peak_row$year - 11, y = peak_row$lead_ng_g * 0.58,
           label = sprintf("1950 spike\n%.2f ng/g (%.0fx pre-industrial)",
                           peak_row$lead_ng_g,
                           peak_row$lead_ng_g / pre_industrial),
           family = "jetbrains", fontface = "bold", size = 3.3,
           color = "#FFFFFF", hjust = 1, lineheight = 0.95) +

  geom_segment(data = events,
               aes(x = year, xend = year, y = 0.003, yend = 0.003),
               inherit.aes = FALSE, color = "#94a3b8", linewidth = 0.3) +
  geom_text(data = events,
            aes(x = year, y = 0.0045, label = label),
            inherit.aes = FALSE,
            family = "roboto_condensed", size = 2.8,
            color = "#cbd5e1", hjust = 0.5, vjust = 0, lineheight = 0.9) +

  annotate("label", x = 1775, y = 0.25,
    label = paste0(
      "Even before tetraethyl lead (1923),\n",
      "coal burning had pushed Arctic Pb\n",
      "to ~11x pre-industrial by 1900-1920.\n",
      "The 1950 spike hit 44x baseline.\n",
      "Post-regulation: down to ~3x.\n",
      "Reduced, but never restored."),
    family = "outfit", fontface = "bold", size = 3.6,
    color = "#FFFFFF", fill = "#0a0e17EE",
    label.r = unit(0.1, "lines"),
    label.padding = unit(0.5, "lines"),
    lineheight = 1.2, hjust = 0, vjust = 1) +

  scale_x_continuous(breaks = seq(1780, 2000, 20),
                     expand = expansion(mult = c(0.01, 0.01))) +
  scale_y_log10(
    breaks = c(0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5),
    labels = function(y) sprintf("%g", y),
    limits = c(0.003, 0.6),
    expand = expansion(mult = c(0, 0))) +

  labs(
    title = "230 Years of Lead Pollution in the Arctic",
    subtitle = paste0(
      "Annual lead concentration in the ACT2 Greenland ice core, 1772\u20132003 \u2014 log scale.\n",
      "Orange line: 10-year rolling mean. Dashed grey: late-18th-century baseline."
    ),
    x = NULL,
    y = "Pb concentration (ng per g of ice)",
    caption = paste0(
      "Data: McConnell, J.R. & Edwards, R. (2008). Coal burning leaves toxic heavy metal legacy in the Arctic. ",
      "PNAS 105(34): 12140\u201312144. doi:10.1073/pnas.0803564105.\n",
      "NOAA/WDS Paleoclimatology, Data Contribution Series 2008-079. ACT2 ice core (66\u00b0N, 45.2\u00b0W, 2,410 m).\n",
      "#30DayChartChallenge \u00b7 Day 21: Historical \u00b7 Sharfudeen Yasar Arafath"
    )
  ) +

  theme_minimal(base_family = "roboto_condensed") +
  theme(
    plot.title = element_text(family = "outfit", face = "bold", size = 28,
      color = "#FFFFFF", margin = margin(t = 10, b = 5)),
    plot.subtitle = element_text(size = 14, color = txt_dim,
      lineheight = 1.3, margin = margin(b = 20)),
    plot.caption = element_text(size = 10, hjust = 0, color = txt_cap,
      lineheight = 1.5, margin = margin(t = 20)),
    axis.text = element_text(size = 11, color = txt_dim, family = "jetbrains"),
    axis.title.y = element_text(size = 12, color = txt_dim,
      margin = margin(r = 12)),
    panel.grid.major = element_line(color = grid_col, linewidth = 0.15),
    panel.grid.minor = element_blank(),
    plot.background = element_rect(fill = bg, color = NA),
    panel.background = element_rect(fill = bg, color = NA),
    plot.margin = margin(15, 25, 15, 15)
  )

p

In [ ]:
# — save ----------------------------------------------------------------------

ggsave("../../chart/day_21_historical.png",
       plot = p, width = 13, height = 10, dpi = 300, bg = bg)

cat("Done \u2014 saved to chart/day_21_historical.png\n")